# 18 - Gemini Annotation Datasets

Reuters ve SP500 verilerini Gemini ile etiketler.

**Çalışma mantığı**
- Gemini/API hatası olursa program DURMAZ.
- Hatalı kayıt başarılı cevap gelene kadar otomatik olarak yeniden denenir.
- Hata sonrası bekleme: 5 → 10 → 20 → 40 → 80 → 120 sn (120 sn'de sabitlenir).
- 429 (rate limit) hatasında API'nin önerdiği `retryDelay` değeri de dikkate alınır.
- İstekler arası minimum `REQUEST_INTERVAL` saniye beklenir (free tier 5 RPM için 13 sn).
- Geçerli Gemini cevabı gelir gelmez `gemini_label` yazılır.
- **Her başarılı kayıt hemen checkpoint'e kaydedilir.**
- Reuters ve SP500 checkpoint dosyaları birbirinden ayrıdır.
- Notebook yeniden çalıştırılırsa başarılı kayıtlar korunur ve kalan kayıtlar kaldığı yerden devam eder.
- Checkpoint klasörü: `db/annotations/gemini/checkpoints/`


# 1. Kurulum

Gerekli paketler.


In [1]:
%pip install -q -U google-genai openpyxl pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Import ve genel ayarlar


In [2]:
from pathlib import Path
import os
import time
import re
from getpass import getpass

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# 3. Proje yolları


In [3]:
NOTEBOOK_DIR = Path.cwd()

PROJECT_DIR = (
    NOTEBOOK_DIR.parent
    if NOTEBOOK_DIR.name == "app"
    else NOTEBOOK_DIR
)

DATA_DIR = PROJECT_DIR / "db"
annotations_dir = DATA_DIR / "annotations"

print("=" * 100)
print("PROJE YOLLARI")
print("=" * 100)
print(f"PROJECT_DIR : {PROJECT_DIR}")
print(f"DATA_DIR    : {DATA_DIR}")
print(f"ANNOTATIONS : {annotations_dir}")

PROJE YOLLARI
PROJECT_DIR : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis
DATA_DIR    : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db
ANNOTATIONS : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations


# 4. Reuters veri setini yükle


In [4]:
reuters_path = (
    annotations_dir
    / "reuters_5000"
    / "REUTERS_annotation_all_clean.csv"
)

if not reuters_path.exists():
    raise FileNotFoundError(
        f"Reuters dosyası bulunamadı:\n{reuters_path}"
    )

reuters_df = pd.read_csv(
    reuters_path,
    encoding="utf-8-sig"
)

print("=" * 100)
print("REUTERS")
print("=" * 100)
print(f"Dosya : {reuters_path}")
print(f"Satır : {len(reuters_df)}")
print(f"Kolon : {len(reuters_df.columns)}")
print("\nKolonlar:")
print(list(reuters_df.columns))

REUTERS
Dosya : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\reuters_5000\REUTERS_annotation_all_clean.csv
Satır : 5000
Kolon : 18

Kolonlar:
['annotation_id', 'date', 'text_en', 'text_tr', 'finbert_score', 'finbert_label', 'finbert_confidence', 'chatgpt_label', 'chatgpt_score', 'chatgpt_confidence', 'chatgpt_reason_tr', 'final_label', 'finbert_correctness', 'review_needed', 'row_note', 'batch_no', 'n_words', 'text_len']


# 5. SP500 batch dosyalarını yükle


In [5]:
sp500_batches_dir = (
    annotations_dir
    / "sp500_human_review_batches"
)

if not sp500_batches_dir.exists():
    raise FileNotFoundError(
        f"SP500 klasörü bulunamadı:\n{sp500_batches_dir}"
    )

sp500_dfs = []

for batch_file in sorted(
    sp500_batches_dir.glob("SP500_annotation_batch_*.xlsx")
):
    try:
        batch_df = pd.read_excel(
            batch_file,
            engine="openpyxl"
        )
        sp500_dfs.append(batch_df)

        print(
            f"✓ {batch_file.name} "
            f"| {len(batch_df)} satır "
            f"| {len(batch_df.columns)} kolon"
        )

    except Exception as e:
        raise RuntimeError(
            f"SP500 dosyası okunamadı:\n{batch_file}\n\nHata: {e}"
        ) from e

if not sp500_dfs:
    raise RuntimeError(
        "Hiçbir SP500 batch dosyası okunamadı."
    )

sp500_combined = pd.concat(
    sp500_dfs,
    ignore_index=True
)

print("\n" + "=" * 100)
print("SP500 BİRLEŞTİRİLMİŞ")
print("=" * 100)
print(f"Satır : {len(sp500_combined)}")
print(f"Kolon : {len(sp500_combined.columns)}")
print("\nKolonlar:")
print(list(sp500_combined.columns))

✓ SP500_annotation_batch_001.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_002.xlsx | 10 satır | 17 kolon
✓ SP500_annotation_batch_004.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_005.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_006.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_007.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_008.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_009.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_010.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_011.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_012.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_013.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_014.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_015.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_016.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_017.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_018.xlsx | 10 satır | 15 kolon
✓ SP500_annotation_batch_019.xlsx | 10 satır | 1

# 6. Reuters'i 3'er sütun halinde incele

Bu hücre sadece veri setini analiz eder. **Henüz Gemini çağrısı yapmaz.**


In [6]:
print("=" * 100)
print("REUTERS - 3'ER SÜTUN")
print("=" * 100)
print(f"Satır: {len(reuters_df)} | Kolon: {len(reuters_df.columns)}")

for i in range(0, len(reuters_df.columns), 3):
    print(f"\nKolonlar: {list(reuters_df.columns[i:i+3])}")
    display(
        reuters_df.iloc[:, i:i+3].head(5)
    )


REUTERS - 3'ER SÜTUN
Satır: 5000 | Kolon: 18

Kolonlar: ['annotation_id', 'date', 'text_en']


,annotation_id,date,text_en
0,REUTERS_ANN_00001,2007-04-12,s.africa watchdog to probe gold fields bid report
1,REUTERS_ANN_00002,2007-04-26,new barbie girls sashay into view with mp-3
2,REUTERS_ANN_00003,2006-12-17,"stocks await data, mergers and santa"
3,REUTERS_ANN_00004,2007-01-03,canadian dollar falls to 9-mth lows vs u.s. dollar
4,REUTERS_ANN_00005,2007-02-05,"techs buckle under microsoft, wal-mart helps dow"



Kolonlar: ['text_tr', 'finbert_score', 'finbert_label']


,text_tr,finbert_score,finbert_label
0,Güney Afrika rekabet kurumu Gold Fields teklifi raporunu inceleyecek,-1,negative
1,Yeni Barbie Girls MP3 ile sahneye çıkıyor,0,neutral
2,"Hisseler veri, birleşmeler ve Noel Baba rallisini bekliyor",0,neutral
3,Kanada doları ABD doları karşısında 9 ayın en düşük seviyesine düştü,-1,negative
4,"Teknoloji hisseleri Microsoft baskısıyla çöktü, Wal-Mart Dow’a yardım etti",-1,negative



Kolonlar: ['finbert_confidence', 'chatgpt_label', 'chatgpt_score']


...

,chatgpt_confidence,chatgpt_reason_tr,final_label
0,0.86,Bir satın alma teklifinin rekabet kurumu tarafından incelenmesi düzenleyici belirsizlik ve işlem riski yaratır.,negative
1,0.76,Yeni MP3 özellikli ürün lansmanı ürün yeniliği ve satış potansiyeli açısından sınırlı olumlu sinyal verir.,positive
2,0.82,Piyasa bekleyişi anlatılıyor; veri veya birleşmelerin net sonucu başlıkta yok.,neutral
3,0.98,Para biriminin 9 ayın en düşük seviyesine gerilemesi güçlü negatif döviz hareketidir.,negative
4,0.86,Wal-Mart desteği olumlu olsa da teknoloji hisselerindeki baskı ve 'buckle' ifadesi baskın negatif piyasa sinyali verir.,negative



Kolonlar: ['finbert_correctness', 'review_needed', 'row_note']


,finbert_correctness,review_needed,row_note
0,same,no,FinBERT etiketi bağlamla uyumlu; 'watchdog to probe' ifadesi düzenleyici soruşturma/inceleme riskini negatif yakalamış.
1,different,yes,FinBERT bunu nötr görmüş olabilir çünkü başlık finansal metrik içermiyor; ancak yeni ürün tanıtımı pozitif ürün/growth sinyalidir.
2,same,no,FinBERT etiketi bağlamla uyumlu; 'await' bekle-gör ve yönsüz piyasa tonudur.
3,same,no,FinBERT etiketi bağlamla uyumlu; 'falls to lows' net negatif.
4,same,no,FinBERT etiketi bağlamla uyumlu; tech weakness başlığın negatif tarafını yakalamış.



Kolonlar: ['batch_no', 'n_words', 'text_len']


,batch_no,n_words,text_len
0,1,8,49
1,1,8,43
2,1,6,36
3,1,9,50
4,1,7,48


# 7. SP500'ü 3'er sütun halinde incele


In [7]:
print("=" * 100)
print("SP500 - 3'ER SÜTUN")
print("=" * 100)
print(f"Satır: {len(sp500_combined)} | Kolon: {len(sp500_combined.columns)}")

for i in range(0, len(sp500_combined.columns), 3):
    print(f"\nKolonlar: {list(sp500_combined.columns[i:i+3])}")
    display(
        sp500_combined.iloc[:, i:i+3].head(5)
    )


SP500 - 3'ER SÜTUN
Satır: 1067 | Kolon: 32

Kolonlar: ['annotation_id', 'sample_id', 'date']


,annotation_id,sample_id,date
0,SP500_ANN_0001,SP500_HEAD_000004,2008-01-03
1,SP500_ANN_0002,SP500_HEAD_016918,2023-12-15
2,SP500_ANN_0003,SP500_HEAD_013289,2023-03-20
3,SP500_ANN_0004,SP500_HEAD_017167,2024-01-04
4,SP500_ANN_0005,SP500_HEAD_007040,2019-09-18



Kolonlar: ['text_en', 'text_tr', 'finbert_label']


,text_en,text_tr,finbert_label
0,"U.S. Stocks Higher After Economic Data, Monsanto Outlook",ABD hisseleri ekonomik veriler ve Monsanto görünümü sonrası yükseldi.,positive
1,Stock Market Outlook 2024: Rare Bullish Signal Says S&P 500 Will Soar 20%,2024 borsa görünümü: Nadir bir boğa sinyali S&P 500'ün %20 yükseleceğini söylüyor.,positive
2,Federal Reserve Rate Hike Odds Grow As Bank-Crisis Fears Ebb; S&P 500 Rises,Banka krizi korkuları azalırken Fed faiz artırımı olasılığı yükseliyor; S&P 500 yükseliyor.,negative
3,"US Stock Market Closing: Dow Jones, S&P 500 Retreat As Traders Assess Fed Meeting Minutes, Nasdaq Plunges 1.2","ABD borsa kapanışı: Fed tutanakları değerlendirilirken Dow Jones ve S&P 500 geriledi, Nasdaq %1,2 düştü.",negative
4,"Deutsche Bank: S&P 500 13% Overvalued, Recession Coming","Deutsche Bank: S&P 500 %13 aşırı değerli, resesyon geliyor.",negative



Kolonlar: ['finbert_confidence', 'chatgpt_label', 'chatgpt_confidence']


...

,Unnamed: 6,Unnamed: 7,Unnamed: 8
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN



Kolonlar: ['Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11']


,Unnamed: 9,Unnamed: 10,Unnamed: 11
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,NaN,NaN,NaN
4,NaN,NaN,NaN



Kolonlar: ['SP500 Annotation Batch 107', 'SP500 Annotation Batch 109']


,SP500 Annotation Batch 107,SP500 Annotation Batch 109
0,NaN,NaN
1,NaN,NaN
2,NaN,NaN
3,NaN,NaN
4,NaN,NaN


# 8. Veri seti kalite ve etiket analizi


In [8]:
print("=" * 100)
print("VERİ SETİ ANALİZİ")
print("=" * 100)

print("\nREUTERS BOŞ DEĞERLER:")
print(reuters_df.isna().sum())

print("\nREUTERS CHATGPT LABEL DAĞILIMI:")
if "chatgpt_label" in reuters_df.columns:
    print(reuters_df["chatgpt_label"].value_counts(dropna=False))
else:
    print("chatgpt_label kolonu yok.")

print("\nSP500 BOŞ DEĞERLER:")
print(sp500_combined.isna().sum())

print("\nSP500 CHATGPT LABEL DAĞILIMI:")
if "chatgpt_label" in sp500_combined.columns:
    print(sp500_combined["chatgpt_label"].value_counts(dropna=False))
else:
    print("chatgpt_label kolonu yok.")

print("\nTEXT KOLON KONTROLÜ:")
print("Reuters text_en var mı? ->", "text_en" in reuters_df.columns)
print("SP500 text_en var mı?  ->", "text_en" in sp500_combined.columns)

VERİ SETİ ANALİZİ

REUTERS BOŞ DEĞERLER:
annotation_id             0
date                      0
text_en                   0
text_tr                   0
finbert_score             0
finbert_label             0
finbert_confidence        0
chatgpt_label             0
chatgpt_score           350
chatgpt_confidence        0
chatgpt_reason_tr         0
final_label               0
finbert_correctness    1300
review_needed          1454
row_note               1856
batch_no                  0
n_words                   0
text_len                  0
dtype: int64

REUTERS CHATGPT LABEL DAĞILIMI:
chatgpt_label
positive    2275
negative    1579
neutral     1146
Name: count, dtype: int64

SP500 BOŞ DEĞERLER:
annotation_id                   37
sample_id                       37
date                            37
text_en                         37
text_tr                         37
finbert_label                   37
finbert_confidence              37
chatgpt_label                   37
chatgpt_confidenc

# 9. Gemini için kullanılacak aday veri setini hazırla

Bu hücre de sadece hazırlık yapar; API çağrısı yoktur.


In [9]:
datasets_for_gemini = []

# SP500
if (
    not sp500_combined.empty
    and "text_en" in sp500_combined.columns
    and "chatgpt_label" in sp500_combined.columns
):
    sp500_cols = [
        c
        for c in [
            "annotation_id",
            "sample_id",
            "date",
            "text_en",
            "chatgpt_label",
        ]
        if c in sp500_combined.columns
    ]

    sp500_cleaned = (
        sp500_combined[sp500_cols]
        .copy()
    )

    sp500_cleaned = sp500_cleaned[
        sp500_cleaned["text_en"].notna()
        &
        sp500_cleaned["chatgpt_label"].notna()
    ].copy()

    sp500_gemini = pd.DataFrame({
        "source_dataset": "sp500_human_review",
        "source_row_index": range(len(sp500_cleaned)),
        "annotation_id": (
            sp500_cleaned["annotation_id"]
            .astype(str)
            .str.strip()
            .values
        ),
        "text": (
            sp500_cleaned["text_en"]
            .astype(str)
            .str.strip()
            .values
        ),
        "chatgpt_label": (
            sp500_cleaned["chatgpt_label"]
            .astype(str)
            .str.strip()
            .str.lower()
            .values
        ),
    })

    sp500_gemini = sp500_gemini[
        sp500_gemini["text"].str.len() > 0
    ].copy()

    datasets_for_gemini.append(sp500_gemini)

# Reuters
reuters_gemini = pd.DataFrame({
    "source_dataset": "reuters_annotation_clean",
    "source_row_index": reuters_df.index,
    "annotation_id": (
        reuters_df["annotation_id"]
        .astype(str)
        .str.strip()
        .values
    ),
    "text": (
        reuters_df["text_en"]
        .astype(str)
        .str.strip()
        .values
    ),
    "chatgpt_label": (
        reuters_df["chatgpt_label"]
        .astype(str)
        .str.strip()
        .str.lower()
        .values
    ),
})

reuters_gemini = reuters_gemini[
    reuters_gemini["chatgpt_label"].notna()
    &
    (~reuters_gemini["chatgpt_label"].isin([
        "nan", "none", ""
    ]))
    &
    (reuters_gemini["text"].str.len() > 0)
].copy()

datasets_for_gemini.append(reuters_gemini)

if not datasets_for_gemini:
    raise RuntimeError(
        "Gemini için hiç veri oluşturulamadı."
    )

datasets_for_gemini = pd.concat(
    datasets_for_gemini,
    ignore_index=True
)

datasets_for_gemini["gemini_label"] = pd.NA

print("=" * 100)
print("GEMINI ADAY DATASET")
print("=" * 100)
print(f"Toplam satır : {len(datasets_for_gemini)}")

print("\nDataset dağılımı:")
print(
    datasets_for_gemini["source_dataset"]
    .value_counts()
)

print("\nChatGPT label dağılımı:")
print(
    datasets_for_gemini["chatgpt_label"]
    .value_counts(dropna=False)
)

GEMINI ADAY DATASET
Toplam satır : 6030

Dataset dağılımı:
source_dataset
reuters_annotation_clean    5000
sp500_human_review          1030
Name: count, dtype: int64

ChatGPT label dağılımı:
chatgpt_label
positive    2659
negative    1893
neutral     1478
Name: count, dtype: int64


# 10. Gemini aday datasetini yine 3'er sütun halinde kontrol et


In [10]:
print("=" * 100)
print("GEMINI ADAY DATASET - 3'ER SÜTUN")
print("=" * 100)
print(f"Satır: {len(datasets_for_gemini)} | Kolon: {len(datasets_for_gemini.columns)}")

for i in range(0, len(datasets_for_gemini.columns), 3):
    print(f"\nKolonlar: {list(datasets_for_gemini.columns[i:i+3])}")
    display(
        datasets_for_gemini.iloc[:, i:i+3].head(5)
    )


GEMINI ADAY DATASET - 3'ER SÜTUN
Satır: 6030 | Kolon: 6

Kolonlar: ['source_dataset', 'source_row_index', 'annotation_id']


,source_dataset,source_row_index,annotation_id
0,sp500_human_review,0,SP500_ANN_0001
1,sp500_human_review,1,SP500_ANN_0002
2,sp500_human_review,2,SP500_ANN_0003
3,sp500_human_review,3,SP500_ANN_0004
4,sp500_human_review,4,SP500_ANN_0005



Kolonlar: ['text', 'chatgpt_label', 'gemini_label']


,text,chatgpt_label,gemini_label
0,"U.S. Stocks Higher After Economic Data, Monsanto Outlook",positive,<NA>
1,Stock Market Outlook 2024: Rare Bullish Signal Says S&P 500 Will Soar 20%,positive,<NA>
2,Federal Reserve Rate Hike Odds Grow As Bank-Crisis Fears Ebb; S&P 500 Rises,positive,<NA>
3,"US Stock Market Closing: Dow Jones, S&P 500 Retreat As Traders Assess Fed Meeting Minutes, Nasdaq Plunges 1.2",negative,<NA>
4,"Deutsche Bank: S&P 500 13% Overvalued, Recession Coming",negative,<NA>


# 11. Gemini API ayarları

Bu hücrede API bağlantısı hazırlanır. Etiketleme henüz başlamaz.


In [11]:
from pathlib import Path
import os
import time
import re
import json
from getpass import getpass

from google import genai
from google.genai import types

# API key'i notebook'a YAZMA.
api_key = "AQ.Ab8RN6IVVX_2kWKkap7nUDkXllD4tl9j5KY4E5s7FF86rZ3huA";

client = genai.Client(
    api_key=api_key,
    http_options=types.HttpOptions(timeout=60000),
)

MODEL_NAME = "gemini-3.5-flash-lite"
BATCH_SIZE = 5
REQUEST_INTERVAL = 5.0

SENTIMENT_LABELS = {"positive", "negative", "neutral"}

PROMPT_TEMPLATE = """
You are an expert financial sentiment annotator.

Your task is to classify the OVERALL FINANCIAL SENTIMENT of each
financial news headline.

You MUST assign exactly ONE of these labels:

- positive
- negative
- neutral

IMPORTANT: Classify the meaning and financial implication of the
WHOLE headline. Do NOT classify based on individual words.

LABEL DEFINITIONS

positive:
The headline communicates an overall favorable financial or business
development, such as:
- stronger-than-expected performance
- increased profits or revenue
- successful business expansion
- improving financial outlook
- favorable market developments
- positive investor/business implications
- successful deals or developments that are beneficial to a company,
  industry, or market

negative:
The headline communicates an overall unfavorable financial or business
development, such as:
- falling profits or revenue
- weaker-than-expected performance
- deteriorating outlook
- financial losses
- business problems
- negative investor/business implications
- significant risks, failures, or adverse developments

neutral:
The headline is primarily factual, descriptive, procedural, or
informational and does not communicate a clearly favorable or
unfavorable financial implication.

CRITICAL RULES

1. Do NOT classify based only on positive-looking words.

For example:
"factory orders up less than expected"
should NOT automatically be positive.

The phrase "less than expected" changes the financial meaning.

2. Do NOT classify based only on negative-looking words.

For example:
"airlines on rocky road, but recovery remains intact"
contains negative wording, but the overall message is mixed.
If neither direction clearly dominates, classify it as neutral.

3. Consider comparisons with expectations.

Examples:
"profit rises more than expected"
=> positive

"profit rises less than expected"
=> negative

"orders fall less than expected"
=> positive or neutral depending on the
overall financial implication.

"orders fall more than expected"
=> negative

4. Headlines containing BOTH positive and negative information:

Determine the MAIN financial implication.

If one direction clearly dominates, use that direction.

If the headline genuinely presents balanced or conflicting
information without a clearly dominant direction, use neutral.

NEVER return "mixed".

5. Words such as:
"up", "rise", "growth", "help", "recovery"
do NOT automatically mean positive.

Words such as:
"fall", "drop", "weak", "worry", "rocky"
do NOT automatically mean negative.

Always interpret them in context.

6. Neutral should NOT mean "I am uncertain".

Use neutral when the headline itself is informational or when
positive and negative implications are genuinely balanced.

7. Do not use outside knowledge about the companies or events.
Classify ONLY what can reasonably be inferred from the headline itself.

8. Return exactly one label per input item and preserve the input order.

OUTPUT FORMAT

Return JSON only:

{{"labels": ["positive", "neutral", "negative"]}}

The "labels" array MUST contain exactly one label for every input item.

Allowed labels:
positive
negative
neutral

NEVER return:
mixed
both
uncertain
unknown
or any other label.

ITEMS:

{items}
"""

print("=" * 100)
print("GEMINI API AYARLARI")
print("=" * 100)
print(f"MODEL             : {MODEL_NAME}")
print(f"BATCH_SIZE        : {BATCH_SIZE}")
print(f"REQUEST_INTERVAL  : {REQUEST_INTERVAL} sn")
print("=" * 100)

sample_text = "Revenue for the quarter increased by 15% compared to last year."
test_prompt = PROMPT_TEMPLATE.format(
    items=json.dumps([{"index": 0, "text": sample_text}], ensure_ascii=False)
)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=test_prompt,
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
    ),
)

print("API TEST:", response.text.strip())
print("✓ Gemini API bağlantısı başarılı.")


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


GEMINI API AYARLARI
MODEL             : gemini-3.5-flash-lite
BATCH_SIZE        : 5
REQUEST_INTERVAL  : 5.0 sn
API TEST: {"labels": ["positive"]}
✓ Gemini API bağlantısı başarılı.


# 12. Label yardımcı fonksiyonları


In [12]:
def extract_response_text(response):
    text = getattr(response, "text", None)
    if text:
        return str(text)
    return ""


def parse_labels(response, expected_count):
    raw = extract_response_text(response)

    if not raw:
        raise RuntimeError("Gemini boş cevap döndürdü.")

    try:
        data = json.loads(raw)
    except json.JSONDecodeError as e:
        raise RuntimeError(
            f"Gemini geçerli JSON döndürmedi: {raw!r}"
        ) from e

    labels = data.get("labels") if isinstance(data, dict) else None

    if not isinstance(labels, list):
        raise RuntimeError(
            f"JSON içinde 'labels' listesi yok: {data!r}"
        )

    if len(labels) != expected_count:
        raise RuntimeError(
            f"Beklenen {expected_count} label, gelen {len(labels)}: {data!r}"
        )

    normalized = []

    for label in labels:
        value = str(label).strip().lower()

        if value not in SENTIMENT_LABELS:
            raise RuntimeError(f"Geçersiz label: {label!r}")

        normalized.append(value)

    return normalized


def _extract_retry_delay(err):
    msg = str(err)

    patterns = [
        r"""retryDelay['"]?\s*[:=]\s*['"]?(\d+(?:\.\d+)?)s""",
        r"retry in\s+(\d+(?:\.\d+)?)s",
    ]

    for pattern in patterns:
        m = re.search(pattern, msg, flags=re.IGNORECASE)

        if m:
            return float(m.group(1))

    return None


def ask_gemini_batch(texts):
    items = [
        {"index": i, "text": str(text)}
        for i, text in enumerate(texts)
    ]

    prompt = PROMPT_TEMPLATE.format(
        items=json.dumps(items, ensure_ascii=False)
    )

    retry_count = 0

    while True:
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.0,
                    response_mime_type="application/json",
                ),
            )

            return parse_labels(
                response,
                expected_count=len(texts),
            )

        except KeyboardInterrupt:
            raise

        except Exception as e:
            retry_count += 1

            wait_seconds = min(
                5 * (2 ** (retry_count - 1)),
                120
            )

            suggested = _extract_retry_delay(e)

            if suggested is not None:
                wait_seconds = max(
                    wait_seconds,
                    suggested
                )

            print(
                f"⚠️ Gemini batch hatası | Retry #{retry_count}: {e}"
            )

            print(
                f"   {wait_seconds:.1f} saniye sonra tekrar denenecek..."
            )

            time.sleep(wait_seconds)

### Ücretli API için ayarlar

Bu notebook artık tek tek kayıt göndermek yerine **20 kaydı tek API isteğinde**
etiketler.

- Model: `gemini-2.5-flash-lite`
- Batch: 20 kayıt
- Çıktı: yalnızca `positive / negative / neutral`
- JSON çıktı kontrolü
- Her başarılı batch hemen checkpoint'e yazılır.
- Notebook yarıda kesilirse kaldığı yerden devam eder.
- API/rate-limit hatalarında otomatik retry vardır.
- API key notebook dosyasına kaydedilmez.


# 13. API test

Tek bir örnek gönderilir. Bu test başarısızsa etiketleme başlamaz.


In [13]:
TEST_TEXT = "Revenue for the quarter increased by 15% compared to last year."

print("=" * 100)
print("🧪 GEMINI API TESTİ")
print("=" * 100)

test_labels = ask_gemini_batch([TEST_TEXT])

print("\n" + "=" * 100)
print("✅ API TESTİ BAŞARILI")
print("Sonuç:", test_labels)
print("=" * 100)

🧪 GEMINI API TESTİ

✅ API TESTİ BAŞARILI
Sonuç: ['positive']


# 14. Output / checkpoint yolları

Reuters ve SP500 için ayrı checkpoint ve final dosyaları.


In [14]:
GEMINI_OUTPUT_DIR = DATA_DIR / "annotations" / "gemini"
CHECKPOINT_DIR = GEMINI_OUTPUT_DIR / "checkpoints"

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

REUTERS_CHECKPOINT_FILE = CHECKPOINT_DIR / "reuters_gemini_checkpoint.csv"
SP500_CHECKPOINT_FILE = CHECKPOINT_DIR / "sp500_gemini_checkpoint.csv"

REUTERS_FINAL_FILE = GEMINI_OUTPUT_DIR / "reuters_gemini_labeled.csv"
SP500_FINAL_FILE = GEMINI_OUTPUT_DIR / "sp500_gemini_labeled.csv"
COMBINED_FINAL_FILE = GEMINI_OUTPUT_DIR / "gemini_annotation_labeled.csv"

print("CHECKPOINT DIR :", CHECKPOINT_DIR)
print("REUTERS CHECKPOINT:", REUTERS_CHECKPOINT_FILE)
print("SP500 CHECKPOINT  :", SP500_CHECKPOINT_FILE)
print("REUTERS FINAL     :", REUTERS_FINAL_FILE)
print("SP500 FINAL       :", SP500_FINAL_FILE)
print("COMBINED FINAL    :", COMBINED_FINAL_FILE)


CHECKPOINT DIR : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints
REUTERS CHECKPOINT: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\reuters_gemini_checkpoint.csv
SP500 CHECKPOINT  : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\sp500_gemini_checkpoint.csv
REUTERS FINAL     : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\reuters_gemini_labeled.csv
SP500 FINAL       : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\sp500_gemini_labeled.csv
COMBINED FINAL    : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\gemini_annotation_labeled.csv


# 15. Resume: Reuters ve SP500 checkpoint'lerini ayrı yükle

Her dataset kendi checkpoint'inden devam eder.


In [15]:
def load_checkpoint_into_dataset(dataset, checkpoint_file):
    result = dataset.copy()

    checkpoint_path = Path(checkpoint_file)

    # Checkpoint hiç yoksa → normal şekilde devam et
    if not checkpoint_path.exists():
        print(f"Checkpoint yok: {checkpoint_file}")
        return result

    # Dosya varsa ama boşsa → normal şekilde devam et
    if checkpoint_path.stat().st_size == 0:
        print(f"Checkpoint boş, atlanıyor: {checkpoint_file}")
        return result

    try:
        checkpoint = pd.read_csv(
            checkpoint_path,
            encoding="utf-8-sig",
            dtype=str
        )

    except pd.errors.EmptyDataError:
        print(f"Checkpoint boş, atlanıyor: {checkpoint_file}")
        return result

    required = {
        "source_dataset",
        "source_row_index",
        "annotation_id",
        "text",
        "gemini_label",
    }

    missing = required - set(checkpoint.columns)

    if missing:
        raise ValueError(
            f"Checkpoint kolonları eksik: {missing}"
        )

    # Sadece geçerli Gemini etiketlerini kullan
    checkpoint["gemini_label"] = (
        checkpoint["gemini_label"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    checkpoint = checkpoint[
        checkpoint["gemini_label"].isin(
            ["positive", "negative", "neutral"]
        )
    ].copy()

    if checkpoint.empty:
        print(
            f"Checkpoint'te geçerli Gemini etiketi yok: "
            f"{checkpoint_file}"
        )
        return result

    # annotation_id üzerinden Gemini etiketlerini dataset'e aktar
    checkpoint_map = (
        checkpoint
        .drop_duplicates(
            subset=["annotation_id"],
            keep="last"
        )
        .set_index("annotation_id")["gemini_label"]
    )

    result_ids = (
        result["annotation_id"]
        .astype(str)
    )

    mapped_labels = result_ids.map(
        checkpoint_map
    )

    # Checkpoint'te bulunanları güncelle,
    # geri kalanları mevcut haliyle bırak.
    result["gemini_label"] = (
        mapped_labels
        .combine_first(
            result["gemini_label"]
        )
    )

    print(
        f"Checkpoint yüklendi: {checkpoint_file}"
    )

    print(
        f"Geçerli Gemini etiketi: "
        f"{checkpoint['gemini_label'].notna().sum()}"
    )

    return result
  

reuters_mask = datasets_for_gemini["source_dataset"] == "reuters_annotation_clean"
sp500_mask = datasets_for_gemini["source_dataset"] == "sp500_human_review"

reuters_dataset = datasets_for_gemini[reuters_mask].copy()
sp500_dataset = datasets_for_gemini[sp500_mask].copy()

reuters_dataset = load_checkpoint_into_dataset(
    reuters_dataset,
    REUTERS_CHECKPOINT_FILE
)

sp500_dataset = load_checkpoint_into_dataset(
    sp500_dataset,
    SP500_CHECKPOINT_FILE
)

datasets_for_gemini = pd.concat(
    [sp500_dataset, reuters_dataset],
    ignore_index=True
)

print(f"Reuters : {len(reuters_dataset)} satır")
print(f"SP500   : {len(sp500_dataset)} satır")


Checkpoint yok: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\reuters_gemini_checkpoint.csv
Checkpoint yüklendi: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\sp500_gemini_checkpoint.csv
Geçerli Gemini etiketi: 185
Reuters : 5000 satır
SP500   : 1030 satır


# 16. Kaç satır kaldığını hesapla


In [16]:
gemini_normalized = (
    datasets_for_gemini["gemini_label"]
    .astype("string")
    .str.strip()
    .str.lower()
)

unlabeled_mask = (
    gemini_normalized.isna()
    |
    gemini_normalized.isin([
        "",
        "nan",
        "none",
        "<na>",
        "hata",
        "error",
    ])
)

unlabeled_indices = (
    datasets_for_gemini[
        unlabeled_mask
    ]
    .index
    .tolist()
)

total_rows = len(
    datasets_for_gemini
)

already_labeled_count = (
    total_rows
    - len(unlabeled_indices)
)

print("=" * 100)
print("ETİKETLEME DURUMU")
print("=" * 100)
print(f"Toplam satır   : {total_rows}")
print(f"Etiketli       : {already_labeled_count}")
print(f"İşlenecek      : {len(unlabeled_indices)}")

remaining_seconds = (
    len(unlabeled_indices)
    * REQUEST_INTERVAL
)

print(
    f"\nRate ayarı     : "
    f"{60 / REQUEST_INTERVAL:.2f} RPM"
)

print(
    f"Tahmini minimum: "
    f"{remaining_seconds / 3600:.2f} saat"
)

ETİKETLEME DURUMU
Toplam satır   : 6030
Etiketli       : 185
İşlenecek      : 5845

Rate ayarı     : 12.00 RPM
Tahmini minimum: 8.12 saat


# 17. Gemini etiketleme
 **Otomatik retry sistemi aktif.** Bir Gemini/API hatasında aynı batch tekrar denenir.

Her başarılı batch sonrası checkpoint kaydedilir.
Ayrıca Gemini etiketleri mevcut `chatgpt_label` ile karşılaştırılır ve her batch sonrası hem batch uyumu hem kümülatif uyum yüzdesi ekranda gösterilir.

**Not:** Bu yüzde Gemini'nin doğruluğu değildir; Gemini ile mevcut ChatGPT etiketleri arasındaki uyum oranıdır.


In [17]:
# 17. Gemini etiketleme - BATCH

def save_dataset_checkpoint(df, dataset_name):
    if dataset_name == "reuters_annotation_clean":
        path = REUTERS_CHECKPOINT_FILE
    elif dataset_name == "sp500_human_review":
        path = SP500_CHECKPOINT_FILE
    else:
        raise ValueError(f"Bilinmeyen dataset: {dataset_name}")

    save_retry = 0
    while True:
        try:
            df.to_csv(path, index=False, encoding="utf-8-sig")
            return path
        except KeyboardInterrupt:
            raise
        except Exception as e:
            save_retry += 1
            wait_seconds = min(5 * (2 ** (save_retry - 1)), 120)
            print(f"Checkpoint yazılamadı | Retry #{save_retry}: {e}")
            time.sleep(wait_seconds)


dataset_jobs = [
    ("sp500_human_review", sp500_dataset),
    ("reuters_annotation_clean", reuters_dataset),
]

for dataset_name, current_df in dataset_jobs:

    current_df = current_df.copy()

    normalized = (
        current_df["gemini_label"]
        .astype("string")
        .str.strip()
        .str.lower()
    )

    remaining_indices = current_df.index[
        ~normalized.isin(["positive", "negative", "neutral"])
    ].tolist()

    total_batches = (
        len(remaining_indices) + BATCH_SIZE - 1
    ) // BATCH_SIZE

    # Gemini vs mevcut ChatGPT etiketleri için kümülatif uyum.
    total_compared = 0
    total_same = 0

    print("\n" + "#" * 110)
    print(f"BAŞLIYOR: {dataset_name}")
    print(f"Toplam kayıt   : {len(current_df)}")
    print(f"Zaten etiketli : {len(current_df) - len(remaining_indices)}")
    print(f"Etiketlenecek  : {len(remaining_indices)}")
    print(f"API batch sayısı: {total_batches}")
    print("#" * 110)

    last_call_ts = 0.0

    for batch_no, start in enumerate(
        range(0, len(remaining_indices), BATCH_SIZE),
        start=1
    ):
        batch_indices = remaining_indices[start:start + BATCH_SIZE]

        texts = [
            str(current_df.loc[idx, "text"])
            for idx in batch_indices
        ]

        elapsed = time.time() - last_call_ts
        if elapsed < REQUEST_INTERVAL:
            time.sleep(REQUEST_INTERVAL - elapsed)

        print(
            f"\n[{batch_no}/{total_batches}] "
            f"{dataset_name} | {len(batch_indices)} kayıt"
        )

        labels = ask_gemini_batch(texts)
        last_call_ts = time.time()

        batch_same = 0
        batch_compared = 0

        for idx, label in zip(batch_indices, labels):
            current_df.loc[idx, "gemini_label"] = label

            # Gemini ile mevcut ChatGPT etiketini karşılaştır.
            chatgpt_label = str(
                current_df.loc[idx, "chatgpt_label"]
            ).strip().lower()

            if chatgpt_label in SENTIMENT_LABELS:
                batch_compared += 1
                total_compared += 1

                if label == chatgpt_label:
                    batch_same += 1
                    total_same += 1

            annotation_id = str(
                current_df.loc[idx, "annotation_id"]
            )

            mask = (
                (datasets_for_gemini["source_dataset"] == dataset_name)
                & (
                    datasets_for_gemini["annotation_id"].astype(str)
                    == annotation_id
                )
            )

            datasets_for_gemini.loc[
                mask,
                "gemini_label"
            ] = label

        batch_agreement = (
            (batch_same / batch_compared) * 100
            if batch_compared
            else 0.0
        )

        total_agreement = (
            (total_same / total_compared) * 100
            if total_compared
            else 0.0
        )

        checkpoint_path = save_dataset_checkpoint(
            current_df,
            dataset_name
        )

        print(
            f"✅ Batch tamamlandı: {len(batch_indices)} kayıt"
        )
        print(
            f"📊 ChatGPT ile bu batch uyumu: "
            f"{batch_agreement:.1f}% "
            f"({batch_same}/{batch_compared})"
        )
        print(
            f"📊 ChatGPT ile toplam uyum: "
            f"{total_agreement:.1f}% "
            f"({total_same}/{total_compared})"
        )
        print(f"💾 Checkpoint: {checkpoint_path}")

print("\n" + "=" * 110)
print("✅ TÜM ETİKETLEME TAMAMLANDI.")
print("=" * 110)



##############################################################################################################
BAŞLIYOR: sp500_human_review
Toplam kayıt   : 1030
Zaten etiketli : 185
Etiketlenecek  : 845
API batch sayısı: 169
##############################################################################################################

[1/169] sp500_human_review | 5 kayıt
✅ Batch tamamlandı: 5 kayıt
📊 ChatGPT ile bu batch uyumu: 80.0% (4/5)
📊 ChatGPT ile toplam uyum: 80.0% (4/5)
💾 Checkpoint: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\sp500_gemini_checkpoint.csv

[2/169] sp500_human_review | 5 kayıt
✅ Batch tamamlandı: 5 kayıt
📊 ChatGPT ile bu batch uyumu: 80.0% (4/5)
📊 ChatGPT ile toplam uyum: 80.0% (8/10)
💾 Checkpoint: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\checkpoints\sp500_gemini_checkpoint.csv

[3/169] sp500_human_review | 5 kayıt
✅ Batch tamamlandı: 5 kayıt
📊 ChatG

# 18. Final dosyaları ve consensus

Reuters ve SP500 ayrı final dosyalarına kaydedilir.


In [18]:
reuters_final = datasets_for_gemini[
    datasets_for_gemini["source_dataset"] == "reuters_annotation_clean"
].copy()

sp500_final = datasets_for_gemini[
    datasets_for_gemini["source_dataset"] == "sp500_human_review"
].copy()

reuters_final.to_csv(
    REUTERS_FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

sp500_final.to_csv(
    SP500_FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

datasets_for_gemini.to_csv(
    COMBINED_FINAL_FILE,
    index=False,
    encoding="utf-8-sig"
)

print("✅ Reuters final:", REUTERS_FINAL_FILE)
print("✅ SP500 final  :", SP500_FINAL_FILE)
print("✅ Birleşik final:", COMBINED_FINAL_FILE)

valid_labels = {"positive", "negative", "neutral"}

chatgpt_norm = (
    datasets_for_gemini["chatgpt_label"]
    .astype("string")
    .str.lower()
    .str.strip()
)

gemini_norm = (
    datasets_for_gemini["gemini_label"]
    .astype("string")
    .str.lower()
    .str.strip()
)

consensus_mask = (
    chatgpt_norm.isin(valid_labels)
    & gemini_norm.isin(valid_labels)
    & (chatgpt_norm == gemini_norm)
)

consensus_df = datasets_for_gemini[consensus_mask].copy()

print("\n" + "=" * 100)
print("CONSENSUS")
print("=" * 100)
print("Toplam:", len(datasets_for_gemini))
print("Consensus:", len(consensus_df))

if len(datasets_for_gemini) > 0:
    print(
        "Consensus rate: %.2f%%"
        % (100 * len(consensus_df) / len(datasets_for_gemini))
    )


✅ Reuters final: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\reuters_gemini_labeled.csv
✅ SP500 final  : d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\sp500_gemini_labeled.csv
✅ Birleşik final: d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\gemini_annotation_labeled.csv

CONSENSUS
Toplam: 6030
Consensus: 4721
Consensus rate: 78.29%


# 19. Consensus dosyalarını kaydet


In [19]:
consensus_output_dir = (
    DATA_DIR
    / "processed"
    / "gemini_consensus_2024"
)

consensus_output_dir.mkdir(
    parents=True,
    exist_ok=True
)

consensus_results = []

for source_name in (
    datasets_for_gemini[
        "source_dataset"
    ]
    .dropna()
    .unique()
):

    source_consensus = (
        consensus_df[
            consensus_df[
                "source_dataset"
            ] == source_name
        ]
        .copy()
    )

    if source_consensus.empty:
        print(
            f"⚠️ {source_name}: 0 consensus satır"
        )
        continue

    consensus_results.append(
        source_consensus
    )

    output_path = (
        consensus_output_dir
        /
        f"{source_name}_consensus.csv"
    )

    source_consensus.to_csv(
        output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"✅ {source_name}: "
        f"{len(source_consensus)} satır"
    )

if consensus_results:

    consensus_combined = pd.concat(
        consensus_results,
        ignore_index=True
    )

    combined_output_path = (
        consensus_output_dir
        /
        "all_consensus_combined.csv"
    )

    consensus_combined.to_csv(
        combined_output_path,
        index=False,
        encoding="utf-8-sig"
    )

    print(
        f"\n✅ Birleşik consensus kaydedildi:"
        f"\n{combined_output_path}"
    )

else:

    print(
        "\n❌ Consensus satırı bulunamadı."
    )

✅ sp500_human_review: 879 satır
✅ reuters_annotation_clean: 3842 satır

✅ Birleşik consensus kaydedildi:
d:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\processed\gemini_consensus_2024\all_consensus_combined.csv


In [26]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score

# ============================================================
# GERÇEK GEMINI FINAL DOSYALARI
# ============================================================

REUTERS_FILE = r"D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\reuters_gemini_labeled.csv"

SP500_FILE = r"D:\serkan.kaymak\financial_sentiment_thesis\financial_sentiment_thesis\db\annotations\gemini\sp500_gemini_labeled.csv"

VALID_LABELS = ["negative", "neutral", "positive"]


def calculate_kappa(file_path, dataset_name):

    df = pd.read_csv(
        file_path,
        encoding="utf-8-sig"
    )

    # Label temizleme
    for col in ["chatgpt_label", "gemini_label"]:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.lower()
        )

    # Sadece iki modelin de geçerli etiket verdiği kayıtlar
    data = df[
        df["chatgpt_label"].isin(VALID_LABELS)
        &
        df["gemini_label"].isin(VALID_LABELS)
    ].copy()

    agreement = (
        data["chatgpt_label"]
        ==
        data["gemini_label"]
    ).mean()

    kappa = cohen_kappa_score(
        data["chatgpt_label"],
        data["gemini_label"],
        labels=VALID_LABELS
    )

    print("\n" + "=" * 90)
    print(dataset_name)
    print("=" * 90)

    print(f"Toplam final kayıt    : {len(df)}")
    print(f"Karşılaştırılan kayıt : {len(data)}")
    print(
        f"Yüzde uyum            : "
        f"{agreement:.4f} ({agreement:.2%})"
    )
    print(
        f"Cohen's Kappa (κ)     : "
        f"{kappa:.4f}"
    )

    # Confusion Matrix
    matrix = pd.crosstab(
        data["chatgpt_label"],
        data["gemini_label"],
        rownames=["ChatGPT"],
        colnames=["Gemini"]
    ).reindex(
        index=VALID_LABELS,
        columns=VALID_LABELS,
        fill_value=0
    )

    print("\nConfusion Matrix:")
    print(matrix)

    # Anlaşmazlıklar
    disagreements = data[
        data["chatgpt_label"]
        != data["gemini_label"]
    ]

    print(
        f"\nAnlaşmazlık sayısı   : "
        f"{len(disagreements)}"
    )

    if not disagreements.empty:

        print("\nAnlaşmazlık dağılımı:")
        print(
            pd.crosstab(
                disagreements["chatgpt_label"],
                disagreements["gemini_label"],
                rownames=["ChatGPT"],
                colnames=["Gemini"]
            )
        )

    return {
        "dataset": dataset_name,
        "total": len(df),
        "compared": len(data),
        "agreement": agreement,
        "kappa": kappa,
        "matrix": matrix,
        "disagreements": disagreements
    }


# ============================================================
# HESAPLA
# ============================================================

reuters_result = calculate_kappa(
    REUTERS_FILE,
    "REUTERS"
)

sp500_result = calculate_kappa(
    SP500_FILE,
    "SP500"
)


REUTERS
Toplam final kayıt    : 5000
Karşılaştırılan kayıt : 5000
Yüzde uyum            : 0.7684 (76.84%)
Cohen's Kappa (κ)     : 0.6560

Confusion Matrix:
Gemini    negative  neutral  positive
ChatGPT                              
negative      1176      390        13
neutral         25     1051        70
positive         4      656      1615

Anlaşmazlık sayısı   : 1158

Anlaşmazlık dağılımı:
Gemini    negative  neutral  positive
ChatGPT                              
negative         0      390        13
neutral         25        0        70
positive         4      656         0

SP500
Toplam final kayıt    : 1030
Karşılaştırılan kayıt : 1030
Yüzde uyum            : 0.8534 (85.34%)
Cohen's Kappa (κ)     : 0.7794

Confusion Matrix:
Gemini    negative  neutral  positive
ChatGPT                              
negative       258       54         2
neutral          8      295        29
positive         0       58       326

Anlaşmazlık sayısı   : 151

Anlaşmazlık dağılımı:
Gemini    negat